# 02 · `gl_engine/errors.py`

## What this file is for

Every way this engine is allowed to fail, in one place — seventy-seven lines.

Read it early, because in this codebase **the refusals are the design**. Most engines treat an error as what happens when something goes wrong. Here, a great deal of deliberate work went into making the engine stop rather than produce a number it can't justify. Knowing the exception types tells you what the engine considers unjustifiable.

**Depends on:** nothing.

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine import errors

for name, obj in vars(errors).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != errors.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

The hierarchy matters more than the list.

In [ ]:
from gl_engine import errors
import inspect

classes = [o for o in vars(errors).values()
           if inspect.isclass(o) and issubclass(o, Exception)]

for c in sorted(classes, key=lambda c: (len(c.__mro__), c.__name__)):
    parents = " <- ".join(b.__name__ for b in c.__mro__[1:3])
    print(f"{c.__name__:<18} {parents}")

## The interesting case

### `ReferToCompany` is deliberately not a load error

This is the distinction the file exists to make. A **load error** means we could not read what ISO filed — that is our problem, and it is a bug. A **referral** means we read ISO perfectly well and ISO's own content says *a human decides this one*. That is not a failure; it is the correct answer.

Conflating them would either hide bugs or turn every legitimate referral into an outage.

In [ ]:
print("ReferToCompany is a LoadError?",
      issubclass(errors.ReferToCompany, errors.LoadError))
print("both descend from EngineError?",
      issubclass(errors.ReferToCompany, errors.EngineError),
      issubclass(errors.LoadError, errors.EngineError))
print()
print(errors.ReferToCompany.__doc__)

### What a referral carries

It names where in ISO's content the refusal came from, so it can be acted on rather than merely reported.

In [ ]:
try:
    raise errors.ReferToCompany(
        "DedFactorProdsCSL row 12",
        None,
        "per-claim deductibles carry a filed factor of 0")
except errors.ReferToCompany as e:
    print(e)

## What it refuses

Nothing — this file raises nothing on its own. It defines the vocabulary the rest of the engine refuses *in*. Every notebook that follows ends with a refusal, and every one of them is a type from this file.

## Try it yourself

1. Grep the engine for each exception type. Which is raised in the most places, and why does that make sense?
2. Find a raise site whose message names a specific ISO artefact. Why does the message matter as much as the type?
3. `AssertionFailure` is raised from only one file — find it. What is different about when it fires?

In [ ]:
# your turn